In [2]:
PREFIX="https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/cohorts/2026/05-monitoring"
!wget $PREFIX/rag_helper.py
!wget $PREFIX/starter.py

--2026-07-14 17:00:59--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/cohorts/2026/05-monitoring/rag_helper.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.108.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1814 (1.8K) [text/plain]
Saving to: ‘rag_helper.py.3’

rag_helper.py.3     100%[===================>]   1.77K  --.-KB/s    in 0s      

2026-07-14 17:01:00 (11.5 MB/s) - ‘rag_helper.py.3’ saved [1814/1814]

--2026-07-14 17:01:00--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/cohorts/2026/05-monitoring/starter.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.109.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting

In [ ]:
from starter import rag

query = "How does the agentic loop keep calling the model until it stops?"
answer = rag.rag(query)
print(answer)

In [2]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor

provider = TracerProvider()
provider.add_span_processor(
    SimpleSpanProcessor(ConsoleSpanExporter())
)
trace.set_tracer_provider(provider)

tracer = trace.get_tracer("llm-zoomcamp")

Q1 and Q3

In [3]:
from starter import RAGBase
class RAGTraced(RAGBase):
    
    def search(self, query):
        with tracer.start_as_current_span("search"):
            return super().search(query)

    def llm(self, prompt):
        with tracer.start_as_current_span("llm"):
            return super().llm(prompt)

    def rag(self, query):
        with tracer.start_as_current_span("rag"):
            return super().rag(query)

In [ ]:
traced_rag = RAGTraced(index=rag.index, llm_client=rag.llm_client)

query = "How does the agentic loop keep calling the model until it stops?"
answer = traced_rag.rag(query)
print(answer)

Q2

In [ ]:
class RAGTraced(RAGBase):
    def search(self, query):
        return super().search(query)

    def llm(self, prompt):
        with tracer.start_as_current_span("llm") as span:
            response = super().llm(prompt)
            usage = response.usage

            span.set_attribute("input_tokens", usage.input_tokens)
            span.set_attribute("output_tokens", usage.output_tokens)
            
            print(f"input_tokens: {usage.input_tokens}")
            print(f"output_tokens: {usage.output_tokens}")
            return response

    def rag(self, query):
        return super().rag(query)

traced_rag = RAGTraced(index=rag.index, llm_client=rag.llm_client)
traced_rag.rag("How does the agentic loop keep calling the model until it stops?")

input_tokens: 7111
output_tokens: 111
{
    "name": "llm",
    "context": {
        "trace_id": "0xdaae83a35fc610b0545053ba6c9280c7",
        "span_id": "0x3450ddf2bceaa51f",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": null,
    "start_time": "2026-07-14T16:54:26.608713Z",
    "end_time": "2026-07-14T16:54:28.189464Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "input_tokens": 7111,
        "output_tokens": 111
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.43.0",
            "service.instance.id": "052dfe15-1f92-4e0b-97cd-5e473bbb8313",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}


'The loop keeps calling the model by checking whether the model returned any `function_call` items.\n\n- If there is a function call, the code runs the tool, appends the tool output to `messages`, and continues.\n- If there are no function calls in that turn, it breaks out of the `while True` loop.\n\nSo the stop condition is:\n\n```python\nif has_function_calls == False:\n    break\n```\n\nIn short: it keeps looping until the model returns a final message with no more tool calls.'

Q4

In [ ]:
import os
import sqlite3
from dotenv import load_dotenv
import pandas as pd
from openai import OpenAI
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor, SpanExporter, SpanExportResult

load_dotenv()
client = OpenAI()

class SQLiteSpanExporter(SpanExporter):
    def __init__(self, db_path="traces.db"):
        self.conn = sqlite3.connect(db_path)
        self.conn.execute("""
            CREATE TABLE IF NOT EXISTS spans (
                name TEXT,
                start_time INTEGER,
                end_time INTEGER,
                input_tokens INTEGER,
                output_tokens INTEGER,
                cost REAL
            )
        """)
        self.conn.commit()

    def export(self, spans):
        for span in spans:
            attrs = dict(span.attributes or {})
            self.conn.execute(
                "INSERT INTO spans VALUES (?, ?, ?, ?, ?, ?)",
                (
                    span.name,
                    span.start_time,
                    span.end_time,
                    attrs.get("input_tokens"),
                    attrs.get("output_tokens"),
                    attrs.get("cost"),
                ),
            )
        self.conn.commit()
        return SpanExportResult.SUCCESS

    def shutdown(self):
        self.conn.close()

    def force_flush(self):
        return True

provider = TracerProvider()
provider.add_span_processor(SimpleSpanProcessor(SQLiteSpanExporter("traces.db")))
trace.set_tracer_provider(provider)
tracer = trace.get_tracer("llm-zoomcamp")

from starter import rag, RAGBase

class RAGTraced(RAGBase):
    def search(self, query):
        with tracer.start_as_current_span("search"):
            results = super().search(query)
            # HACK: Only return the top 1 document to save tokens!
            return results[:1] if results else []

    def llm(self, prompt):
        with tracer.start_as_current_span("llm") as span:
            response = super().llm(prompt)
            usage = response.usage
            span.set_attribute("input_tokens", usage.input_tokens)
            span.set_attribute("output_tokens", usage.output_tokens)
            return response

    def rag(self, query):
        with tracer.start_as_current_span("rag"):
            return super().rag(query)

traced_rag = RAGTraced(index=rag.index, llm_client=new_client)
query = "How does the agentic loop keep calling the model until it stops?"
traced_rag.rag(query)

conn = sqlite3.connect("traces.db")
df = pd.read_sql_query("SELECT DISTINCT name FROM spans", conn)
print("\n--- Spans in Database ---")
print(df)


--- Spans in Database ---
     name
0  search
1     llm
2     rag


Q5

In [2]:
import sqlite3
import pandas as pd

traced_rag.rag("What are the prerequisites for this course?")

conn = sqlite3.connect("traces.db")

sql_query = """
SELECT 
    name, 
    SUM(end_time - start_time) AS total_time
FROM spans 
WHERE name != 'rag' 
GROUP BY name
ORDER BY total_time DESC;
"""

df_durations = pd.read_sql_query(sql_query, conn)

print(df_durations)

     name  total_time
0     llm  7859205680
1  search    22529399


Q6

In [4]:
import sqlite3
import pandas as pd

query = "How does the agentic loop keep calling the model until it stops?"

for i in range(3):
    traced_rag.rag(query)
    print(f"run {i+1}/3")

conn = sqlite3.connect("traces.db")

sql_query = """
SELECT 
    name,
    input_tokens
FROM spans 
WHERE name = 'llm'
"""

df_tokens = pd.read_sql_query(sql_query, conn)

print(df_tokens)

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-5.4-mini in organization org-chuVg18j8T66lBMLkIHpc26M on tokens per min (TPM): Limit 100000, Used 99167, Requested 2504. Please try again in 12h1m52.32s. Visit https://platform.openai.com/account/rate-limits to learn more. You can increase your rate limit by adding a payment method to your account at https://platform.openai.com/account/billing.', 'type': 'tokens', 'param': None, 'code': 'rate_limit_exceeded'}}